In [ ]:
# =====================================================
# Plataforma de Monitoreo Pachamama - Google Colab
# Ejecuta esta celda completa (Ctrl+Enter)
# =====================================================

import os, subprocess, sys, time, re, json, getpass

# 1. Clonar / actualizar repo
REPO_URL = "https://github.com/sdarionicolas-boop/democultivos.git"
APP_DIR  = "/content/pachamama"
if os.path.exists(APP_DIR):
    print("Actualizando repo...")
    subprocess.run(["git", "-C", APP_DIR, "pull"], check=True)
else:
    print("Clonando repo...")
    subprocess.run(["git", "clone", REPO_URL, APP_DIR], check=True)
print("Codigo listo")

# 2. Instalar dependencias
print("Instalando dependencias...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "streamlit", "geopandas", "pandas", "numpy", "matplotlib",
    "folium", "streamlit-folium", "branca", "shapely",
    "earthengine-api", "groq", "pillow", "scipy",
    "requests", "beautifulsoup4", "PyPDF2",
    "scikit-learn", "plotly", "xarray",
], check=True)
print("Dependencias instaladas")

# 3. Ingresar credenciales
print("\n--- CREDENCIALES ---")
groq_key = getpass.getpass("GROQ_API_KEY (Enter para omitir IA): ")
ot_key   = getpass.getpass("OPENTOPOGRAPHY_API_KEY (Enter para omitir DEM): ")

print("\nSubir JSON de cuenta de servicio GEE:")
gee_key_data = gee_client_email = gee_project_id = gee_key_id = gee_client_id = ""
try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        json_filename = list(uploaded.keys())[0]
        with open(json_filename) as f_gee:
            gee_json = json.load(f_gee)
        gee_key_data     = gee_json.get("private_key", "")
        gee_client_email = gee_json.get("client_email", "")
        gee_project_id   = gee_json.get("project_id", "democultivos")
        gee_key_id       = gee_json.get("private_key_id", "")
        gee_client_id    = gee_json.get("client_id", "")
        print("Credenciales GEE cargadas:", gee_client_email)
    else:
        print("Sin GEE - mapa usara valores por defecto")
except Exception as exc:
    print("Error GEE:", exc)

# 4. Escribir secrets.toml
secrets_dir = os.path.join(APP_DIR, ".streamlit")
os.makedirs(secrets_dir, exist_ok=True)
pk_escaped = gee_key_data.replace("\n", "\\n") if gee_key_data else ""
secrets_toml = (
    'GROQ_API_KEY = "' + groq_key + '"\n'
    'OPENTOPOGRAPHY_API_KEY = "' + ot_key + '"\n\n'
)
if gee_key_data:
    secrets_toml += (
        '[gee_service_account]\n'
        'type = "service_account"\n'
        'project_id = "' + gee_project_id + '"\n'
        'private_key_id = "' + gee_key_id + '"\n'
        'client_email = "' + gee_client_email + '"\n'
        'client_id = "' + gee_client_id + '"\n'
        'private_key = "' + pk_escaped + '"\n'
    )
with open(os.path.join(secrets_dir, "secrets.toml"), "w") as f_sec:
    f_sec.write(secrets_toml)
print("secrets.toml escrito")

# 5. Descargar cloudflared
cf_path = "/usr/local/bin/cloudflared"
if not os.path.exists(cf_path):
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", cf_path], check=True)
    os.chmod(cf_path, 0o755)

# 6. Lanzar Streamlit + tunel Cloudflare
print("Lanzando Streamlit...")
subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run",
     os.path.join(APP_DIR, "app.py"),
     "--server.port=8501",
     "--server.headless=true",
     "--server.enableCORS=false"],
    cwd=APP_DIR
)
time.sleep(5)
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
url = None
for _ in range(30):
    line = tunnel_proc.stderr.readline().decode("utf-8", errors="ignore")
    m = re.search(r"https://[a-z0-9\-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break
    time.sleep(1)
if url:
    print("\nApp disponible en:", url)
    print("Compartí este link (valido ~8 horas)")
else:
    print("No se obtuvo URL. Revisa los logs.")


🔑 Configuración de API Key de Groq (obligatoria para usar IA)
✅ API Key configurada

🌍 Configuración opcional de Google Earth Engine
Sube tu archivo JSON de cuenta de servicio de GEE:


Saving democultivos-ce9a27a59d9e.json to democultivos-ce9a27a59d9e.json
✅ Credenciales GEE configuradas

📦 Instalando paquetes Python necesarios...
✅ Dependencias instaladas

🌐 Descargando cloudflared...
✅ cloudflared listo
✅ app.py escrito correctamente

🚀 Iniciando Streamlit...
🌐 Creando túnel Cloudflare...

🎉 APLICACIÓN DISPONIBLE EN: https://sphere-collapse-samuel-monica.trycloudflare.com

Presiona Ctrl+C para detener...
